# System Settings

In [1]:
import asyncio
import sys
import random
import json
import re
import math
import pandas as pd

from pathlib import Path
from datetime import datetime, timedelta
from urllib.parse import quote_plus
from playwright.async_api import async_playwright
import nest_asyncio
import pprint

pp = pprint.PrettyPrinter(indent=2, sort_dicts=False)

In [2]:
# --- Environment Setup ---

# สำหรับ Jupyter Notebook
nest_asyncio.apply()

# สำหรับ Windows
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

print("✅ [Jupyter] Nested Event Loop: Enabled.")

✅ [Jupyter] Nested Event Loop: Enabled.


In [3]:
# --- Config ---

USER_DATA_DIR = "./my_session"
OUTPUT_DIR = "./collected_data"

# Delay request
REQUEST_DELAY_MIN = 2.0
REQUEST_DELAY_MAX = 3.0

# เปลี่ยนเป็น True เฉพาะตอนต้องการ Login / สร้าง session ใหม่
CREATE_SESSION = False

# สร้าง output folder ถ้ายังไม่มี
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"📂 Output folder ready: {OUTPUT_DIR}")

📂 Output folder ready: ./collected_data


In [4]:
# --- Manual Lazada Session Setup ---

async def create_lazada_session():
    """
    ใช้สำหรับ Login และบันทึก Session ของ Browser

    ควรรันเฉพาะกรณี:
    - ใช้งานครั้งแรก
    - Session หมดอายุ
    - Lazada บังคับ Login ใหม่
    """

    async with async_playwright() as p:
        context = await p.chromium.launch_persistent_context(
            user_data_dir=USER_DATA_DIR,
            headless=False,
            args=[
                "--disable-blink-features=AutomationControlled"
            ]
        )

        page = context.pages[0] if context.pages else await context.new_page()

        await page.goto(
            "https://www.lazada.co.th",
            wait_until="domcontentloaded",
            timeout=60000
        )

        print("\n" + "—" * 60)
        print("⚡ [SESSION INITIALIZED] Browser instance is active.")
        print("—" * 60)
        print("📋 ACTION REQUIRED:")
        print(" 1. Login Lazada manually.")
        print(" 2. Confirm default logistics/payment settings if required.")
        print(" 3. Close browser window when finished.")
        print("—" * 60 + "\n")

        # รอจนกว่าผู้ใช้จะปิด browser เอง
        while len(context.pages) > 0:
            await asyncio.sleep(1)

        print(f"✅ [SUCCESS] Session persisted successfully in '{USER_DATA_DIR}'.")

if CREATE_SESSION:
    await create_lazada_session()

# Functions

In [22]:
# --- Common Browser Helpers ---

async def create_browser_context(playwright, block_images=False):
    """
    เปิด Chromium แบบ persistent context เพื่อเก็บ session/cookies

    หมายเหตุ:
    - ห้ามเปิดหลาย browser พร้อมกันโดยใช้ USER_DATA_DIR เดียวกัน
    - ใช้สำหรับ Jupyter / manual scraping
    """

    context = await playwright.chromium.launch_persistent_context(
        user_data_dir=USER_DATA_DIR,
        headless=False,
        viewport={"width": 1366, "height": 768},
        locale="th-TH",
        timezone_id="Asia/Bangkok",
    )

    page = context.pages[0] if context.pages else await context.new_page()

    if block_images:
        async def block_image_route(route):
            await route.abort()

        await page.route("**/*.{png,jpg,jpeg,webp,gif,svg}", block_image_route)

    return context, page


async def human_delay(min_delay=REQUEST_DELAY_MIN, max_delay=REQUEST_DELAY_MAX):
    """
    หน่วงเวลาแบบสุ่ม เพื่อไม่ให้ request ถี่เกินไป
    """

    await asyncio.sleep(random.uniform(min_delay, max_delay))


def make_timestamp():
    """
    ใช้สร้าง timestamp สำหรับชื่อไฟล์
    """

    return datetime.now().strftime("%Y%m%d_%H%M%S")


def make_collected_at():
    """
    ใช้บันทึกเวลาที่ดึงข้อมูล
    """

    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def save_failed_log(failed_rows, prefix):
    """
    บันทึก failed log เป็น Excel
    """

    if not failed_rows:
        return None

    df_failed = pd.DataFrame(failed_rows)
    failed_file = Path(OUTPUT_DIR) / f"{prefix}_{make_timestamp()}.xlsx"
    df_failed.to_excel(failed_file, index=False)

    print(f"⚠️ Failed log saved: {failed_file}")

    return failed_file


def save_debug_html(html, prefix, key="unknown"):
    """
    บันทึก HTML สำหรับ debug กรณีดึงข้อมูลไม่ได้
    """

    debug_file = Path(OUTPUT_DIR) / f"{prefix}_{key}_{make_timestamp()}.html"
    debug_file.write_text(html, encoding="utf-8")

    print(f"⚠️ Debug HTML saved: {debug_file}")

    return debug_file

In [23]:
# --- Manual Lazada Session Setup ---

async def create_lazada_session():
    """
    ใช้สำหรับ Login และบันทึก Session ของ Browser

    ควรรันเฉพาะกรณี:
    - ใช้งานครั้งแรก
    - Session หมดอายุ
    - Lazada บังคับ Login ใหม่
    """

    async with async_playwright() as p:
        context, page = await create_browser_context(p, block_images=False)

        await page.goto(
            "https://www.lazada.co.th",
            wait_until="domcontentloaded",
            timeout=60000
        )

        print("\n" + "—" * 60)
        print("⚡ [SESSION INITIALIZED] Browser instance is active.")
        print("—" * 60)
        print("📋 ACTION REQUIRED:")
        print(" 1. Login Lazada manually.")
        print(" 2. Confirm default logistics/payment settings if required.")
        print(" 3. Close browser window when finished.")
        print("—" * 60 + "\n")

        while len(context.pages) > 0:
            await asyncio.sleep(1)

        print(f"✅ [SUCCESS] Session persisted successfully in '{USER_DATA_DIR}'.")


if CREATE_SESSION:
    await create_lazada_session()

In [24]:
# --- General Helpers ---

def safe_float(val, default=0.0):
    """
    แปลงค่าเป็น float
    """

    if val is None:
        return default

    text = str(val).strip()

    if text == "":
        return default

    try:
        text = (
            text.replace(",", "")
                .replace("฿", "")
                .replace("THB", "")
                .replace("thb", "")
                .strip()
        )

        return float(text)

    except Exception:
        return default


def safe_int(val, default=0):
    """
    แปลงค่าเป็น int
    """

    if val is None:
        return default

    text = str(val).strip()

    if text == "":
        return default

    try:
        text = (
            text.replace(",", "")
                .replace("฿", "")
                .replace("THB", "")
                .replace("thb", "")
                .strip()
        )

        return int(float(text))

    except Exception:
        return default


def parse_sold_count(sold_str):
    """
    แปลงยอดขายจากข้อความ Lazada เป็นตัวเลข

    ตัวอย่าง:
    - "9.4K ชิ้น" -> 9400
    - "1.2k sold" -> 1200
    - "10,000 ชิ้น" -> 10000
    - "2M sold" -> 2000000
    """

    if not sold_str:
        return 0

    s = (
        str(sold_str)
        .lower()
        .replace("ชิ้น", "")
        .replace("ขายแล้ว", "")
        .replace("sold", "")
        .replace(",", "")
        .strip()
    )

    if "k" in s:
        num = safe_float(s.replace("k", ""))
        return int(num * 1000)

    if "m" in s:
        num = safe_float(s.replace("m", ""))
        return int(num * 1_000_000)

    digits = re.sub(r"[^\d]", "", s)

    return safe_int(digits)


def clean_brand(brand):
    """
    ทำความสะอาดชื่อแบรนด์
    """

    if not brand:
        return ""

    brand = str(brand).strip()

    if re.search(r"no\s*brand", brand, re.IGNORECASE):
        return ""

    return brand


def clean_url(url):
    """
    แปลง URL ให้เป็น absolute URL
    """

    if not url:
        return ""

    url = str(url).strip()

    if url.startswith("http"):
        return url

    if url.startswith("//"):
        return f"https:{url}"

    if url.startswith("/"):
        return f"https://www.lazada.co.th{url}"

    return f"https://www.lazada.co.th/{url}"


def clean_full_text(text):
    """
    สำหรับ Description:
    - ยุบบรรทัดว่างซ้ำ
    - ยุบช่องว่างซ้ำ
    - ยังคงโครงสร้างหลายบรรทัดไว้
    """

    if not text:
        return ""

    text = str(text)
    text = re.sub(r"\n\s*\n", "\n", text)
    text = re.sub(r" +", " ", text)

    return text.strip()


def clean_minimal(text):
    """
    สำหรับ Specs / Qualification:
    ยุบทุกอย่างให้เหลือบรรทัดเดียว
    """

    if not text:
        return ""

    return re.sub(r"\s+", " ", str(text)).strip()


def safe_json_loads(raw_text, default=None):
    """
    แปลง JSON แบบปลอดภัย
    """

    if default is None:
        default = {}

    if not raw_text:
        return default

    try:
        return json.loads(raw_text)

    except Exception:
        return default

In [25]:
# --- Date Helper ---

THAI_ABBR_MONTHS = {
    "ม.ค.": "Jan",
    "ก.พ.": "Feb",
    "มี.ค.": "Mar",
    "เม.ย.": "Apr",
    "พ.ค.": "May",
    "มิ.ย.": "Jun",
    "ก.ค.": "Jul",
    "ส.ค.": "Aug",
    "ก.ย.": "Sep",
    "ต.ค.": "Oct",
    "พ.ย.": "Nov",
    "ธ.ค.": "Dec",
}


def thai_date_to_datetime(date_str):
    """
    แปลงวันที่เป็น YYYY-MM-DD

    รองรับ:
    - 05 Apr 2025
    - 05 ม.ค. 2025
    """

    if not date_str:
        return None

    translated = str(date_str).strip()

    for thai_abbr, eng_abbr in THAI_ABBR_MONTHS.items():
        if thai_abbr in translated:
            translated = translated.replace(thai_abbr, eng_abbr)
            break

    try:
        dt_obj = datetime.strptime(translated, "%d %b %Y")
        return dt_obj.strftime("%Y-%m-%d")

    except ValueError:
        return None


def convert_time_interval(interval):
    """
    แปลงเวลาจาก Lazada ให้เป็น YYYY-MM-DD
    """

    if not interval:
        return None

    reference_date = datetime.now()
    interval = str(interval).strip()
    interval_lower = interval.lower()

    if interval in ["วันนี้", "today", "Today"] or interval_lower == "just now":
        return reference_date.strftime("%Y-%m-%d")

    if interval in ["เมื่อวาน", "yesterday", "Yesterday"]:
        return (reference_date - timedelta(days=1)).strftime("%Y-%m-%d")

    parts = interval.split()

    if len(parts) >= 2:
        quantity = safe_int(parts[0], default=None)

        if quantity is not None:
            if any(key in interval for key in ["นาทีที่แล้ว", "minute ago", "minutes ago"]):
                dt = reference_date - timedelta(minutes=quantity)
                return dt.strftime("%Y-%m-%d")

            if any(key in interval for key in ["ชั่วโมงที่แล้ว", "hour ago", "hours ago"]):
                dt = reference_date - timedelta(hours=quantity)
                return dt.strftime("%Y-%m-%d")

            if any(key in interval for key in ["วันที่แล้ว", "day ago", "days ago"]):
                dt = reference_date - timedelta(days=quantity)
                return dt.strftime("%Y-%m-%d")

            if any(key in interval for key in ["สัปดาห์ที่แล้ว", "week ago", "weeks ago"]):
                dt = reference_date - timedelta(weeks=quantity)
                return dt.strftime("%Y-%m-%d")

            if any(key in interval for key in ["เดือนที่แล้ว", "month ago", "months ago"]):
                dt = reference_date - timedelta(days=quantity * 30)
                return dt.strftime("%Y-%m-%d")

            if any(key in interval for key in ["ปีที่แล้ว", "year ago", "years ago"]):
                dt = reference_date - timedelta(days=quantity * 365)
                return dt.strftime("%Y-%m-%d")

    formatted_date = thai_date_to_datetime(interval)

    if formatted_date:
        return formatted_date

    return interval

In [26]:
# --- Page Check Helpers ---

async def handle_page_check(page, label, max_wait_seconds=120):
    """
    ตรวจสอบว่าหน้าเป็น:
    - JSON API ปกติ
    - HTML ปกติ
    - หน้า verification

    หมายเหตุ:
    - ฟังก์ชันนี้ไม่ได้ bypass verification
    - ถ้าเจอ verification จะให้ผู้ใช้จัดการเองใน browser
    """

    start_time = datetime.now()

    while True:
        elapsed = (datetime.now() - start_time).total_seconds()

        if elapsed > max_wait_seconds:
            raise TimeoutError(f"Page check timeout at: {label}")

        try:
            content = (await page.inner_text("body", timeout=5000)).strip()

        except Exception:
            content = ""

        url_lower = page.url.lower()

        if not content:
            await asyncio.sleep(1)
            continue

        if content.startswith("{") or content.startswith("["):
            return content

        has_slider = False

        try:
            has_slider = await page.locator("#nocaptcha, #px-captcha, .btn_slide").count() > 0

        except Exception:
            has_slider = False

        is_verify_url = (
            "verify" in url_lower
            or "punish" in url_lower
            or "captcha" in url_lower
        )

        is_verify_text = any(
            key.lower() in content.lower()
            for key in [
                "verify",
                "verification",
                "captcha",
                "security check",
                "กรุณายืนยัน",
                "ยืนยันตัวตน",
            ]
        )

        if not has_slider and not is_verify_url and not is_verify_text:
            return content

        prompt_msg = (
            f"\n >>>⚠️ [MANUAL CHECK REQUIRED] at {label}\n"
            f" >>> Please complete verification in the browser, then press ENTER to continue..."
        )

        await asyncio.get_event_loop().run_in_executor(None, input, prompt_msg)

        await page.wait_for_load_state("domcontentloaded")
        await asyncio.sleep(1)

In [27]:
# --- Product Info Helpers ---

def extract_product_id_from_url(url):
    """
    ดึง product_id จาก URL Lazada
    """

    if not url:
        return "unknown"

    url = str(url)

    match = re.search(r"-i(\d+)", url)

    if match:
        return match.group(1)

    match = re.search(r"[?&]itemId=(\d+)", url)

    if match:
        return match.group(1)

    return "unknown"


async def safe_inner_text(locator, timeout=3000, default=""):
    """
    ดึง inner_text แบบปลอดภัย
    """

    try:
        return await locator.inner_text(timeout=timeout)

    except Exception:
        return default


async def safe_get_attr(locator, attr_name, timeout=3000, default=""):
    """
    ดึง attribute แบบปลอดภัย
    """

    try:
        value = await locator.get_attribute(attr_name, timeout=timeout)
        return value if value else default

    except Exception:
        return default


async def get_product_info(page, url):
    """
    ดึงข้อมูลพื้นฐานจากหน้าสินค้า:
    - product_id
    - product_name
    - shop_name
    """

    product_id = extract_product_id_from_url(url)

    product_name_selectors = [
        "h1.pdp-mod-product-badge-title-v2",
        "h1.pdp-mod-product-badge-title",
        "h1",
    ]

    product_name = ""

    for selector in product_name_selectors:
        try:
            product_name = await page.locator(selector).first.inner_text(timeout=5000)
            product_name = product_name.strip()

            if product_name:
                break

        except Exception:
            continue

    if not product_name:
        product_name = f"Lazada Product {product_id}"

    shop_name_selectors = [
        ".seller-name-v2__detail-name",
        ".seller-name__detail-name",
        ".pdp-link.pdp-link_size_l.pdp-link_theme_black.seller-name__detail-name",
    ]

    shop_name = ""

    for selector in shop_name_selectors:
        try:
            shop_name = await page.locator(selector).first.inner_text(timeout=5000)
            shop_name = shop_name.strip()

            if shop_name:
                break

        except Exception:
            continue

    if not shop_name:
        shop_name = "Lazada Store"

    return product_id, product_name, shop_name

In [28]:
# --- Product Description Helpers ---

async def wait_and_scroll_until_detail_loaded(page, max_rounds=8):
    """
    Lazada มักโหลด specs / qualification / description หลัง scroll
    ฟังก์ชันนี้ scroll ซ้ำจนกว่าจะเจอข้อมูลจริง
    """

    for round_no in range(1, max_rounds + 1):
        spec_count = await page.locator(".specification-keys .key-li").count()
        qual_count = await page.locator(".pdp-mod-qualification-items .col").count()
        desc_count = await page.locator(".detail-content").count()
        img_count = await page.locator(".detail-content img").count()

        if spec_count > 0 or qual_count > 0 or desc_count > 0 or img_count > 0:
            return {
                "spec_count": spec_count,
                "qual_count": qual_count,
                "desc_count": desc_count,
                "img_count": img_count,
                "loaded": True
            }

        await page.evaluate("window.scrollTo(0, document.body.scrollHeight * 0.35)")
        await asyncio.sleep(1.2)

        await page.evaluate("window.scrollTo(0, document.body.scrollHeight * 0.60)")
        await asyncio.sleep(1.2)

        await page.evaluate("window.scrollTo(0, document.body.scrollHeight * 0.85)")
        await asyncio.sleep(1.2)

        await page.mouse.wheel(0, 1800)
        await asyncio.sleep(1.5)

    return {
        "spec_count": await page.locator(".specification-keys .key-li").count(),
        "qual_count": await page.locator(".pdp-mod-qualification-items .col").count(),
        "desc_count": await page.locator(".detail-content").count(),
        "img_count": await page.locator(".detail-content img").count(),
        "loaded": False
    }


async def extract_product_specs(page):
    """
    ดึง specification table
    """

    specs_data = {}

    spec_rows = page.locator(".specification-keys .key-li")
    spec_count = await spec_rows.count()

    for i in range(spec_count):
        try:
            row = spec_rows.nth(i)

            key = await safe_inner_text(row.locator(".key-title"), timeout=3000)
            value = await safe_inner_text(row.locator(".key-value"), timeout=3000)

            key = clean_minimal(key)
            value = clean_minimal(value)

            if key:
                specs_data[key] = value

        except Exception:
            continue

    return specs_data


async def extract_product_qualification(page):
    """
    ดึง qualification / product highlights
    """

    qual_data = {}

    qual_cols = page.locator(".pdp-mod-qualification-items .col")
    qual_count = await qual_cols.count()

    for i in range(qual_count):
        try:
            col = qual_cols.nth(i)

            q_title = await safe_inner_text(col.locator(".title"), timeout=3000)
            q_content = await safe_inner_text(col.locator(".content"), timeout=3000)

            q_title = clean_minimal(q_title)
            q_content = clean_minimal(q_content)

            if q_title:
                qual_data[q_title] = q_content

        except Exception:
            continue

    return qual_data


async def extract_product_description(page):
    """
    ดึง description text
    """

    description_text = ""

    desc_locator = page.locator(".detail-content")

    if await desc_locator.count() > 0:
        description_text = clean_full_text(
            await safe_inner_text(desc_locator, timeout=8000)
        )

    return description_text


async def extract_product_description_images(page):
    """
    ดึงรูปภาพใน description
    """

    desc_images = []

    img_locators = page.locator(".detail-content img")
    img_count = await img_locators.count()

    for i in range(img_count):
        try:
            img = img_locators.nth(i)

            img_src = (
                await safe_get_attr(img, "src")
                or await safe_get_attr(img, "data-src")
                or await safe_get_attr(img, "data-ks-lazyload")
                or await safe_get_attr(img, "data-lazy")
                or ""
            )

            img_src = str(img_src).strip()

            if img_src and not img_src.startswith("data:"):
                desc_images.append(clean_url(img_src))

        except Exception:
            continue

    desc_images = list(dict.fromkeys(desc_images))

    return desc_images

# Data Scraper

## Product by Shop

In [ ]:
# ชื่อร้านค้าจาก URL ที่ต้องการดึงข้อมูล Ex. https://www.lazada.co.th/mizumi-bomi/?...
SHOP_URL_KEYS = [
    "mizumi-bomi",
    "ing-on-official",
]


async def run_product_by_shop(shop_keys):
    """
    ดึงรายการสินค้าจากหน้าร้าน Lazada ผ่าน ajax=true

    Output:
    - return DataFrame
    - export Excel
    """

    all_products = []
    failed_shops = []

    async with async_playwright() as p:
        print(f"📂 [SESSION] Loading Persistent Context: '{USER_DATA_DIR}'")

        context, page = await create_browser_context(p, block_images=True)

        try:
            for idx, shop_key in enumerate(shop_keys, start=1):
                print("\n" + "-" * 60)
                print(f"[{idx}/{len(shop_keys)}] Processing Shop: {shop_key} | URL: https://www.lazada.co.th/{shop_key}/?q=All-Products&from=wangpu&langFlag=th&pageTypeId=2")

                shop_products = []

                base_url = (
                    f"https://www.lazada.co.th/{shop_key}/"
                    "?ajax=true"
                    "&from=wangpu"
                    "&isFirstRequest=true"
                    "&langFlag=th"
                    "&page=1"
                    "&pageTypeId=2"
                    "&q=All-Products"
                )

                try:
                    await page.goto(base_url, wait_until="domcontentloaded", timeout=60000)

                    raw_text = await handle_page_check(page, f"Shop Meta: {shop_key}")
                    data = safe_json_loads(raw_text)

                    if not data:
                        failed_shops.append({
                            "shop_key": shop_key,
                            "error": "Empty JSON or JSON Decode Error at shop meta"
                        })
                        continue

                    main_info = data.get("mainInfo", {}) or {}

                    total_results = safe_int(main_info.get("totalResults", 0))
                    page_size = safe_int(main_info.get("pageSize", 40), default=40)
                    total_pages = math.ceil(total_results / page_size) if total_results > 0 else 1
                    print(f"   > Total Products Found: {total_results:,} | Total Pages: {total_pages}")

                    for page_no in range(1, total_pages + 1):
                        current_url = base_url.replace("page=1", f"page={page_no}")

                        await page.goto(current_url, wait_until="domcontentloaded", timeout=60000)
                        await human_delay()

                        page_raw = await handle_page_check(page, f"Shop: {shop_key} | Page {page_no}")
                        page_json = safe_json_loads(page_raw)

                        if not page_json:
                            print(f"   ! Page {page_no}: JSON Decode Error")
                            break

                        items = (page_json.get("mods", {}).get("listItems", [])) or []
                        print(f"   > Page {page_no}/{total_pages}: Found {len(items)} products")

                        if not items:
                            break

                        for product in items:
                            price = safe_float(product.get("price"))
                            original_price = safe_float(product.get("originalPrice"))
                            sold_raw = product.get("itemSoldCntShow", "0")

                            icons = product.get("icons", [])
                            if not isinstance(icons, list):
                                icons = []

                            categories = product.get("categories", [])
                            if isinstance(categories, (list, dict)):
                                categories_text = json.dumps(categories, ensure_ascii=False)
                            else:
                                categories_text = str(categories)

                            shop_products.append({
                                "product_id": product.get("itemId", ""),
                                "sku_id": product.get("skuId", ""),
                                "product_name": product.get("name", ""),
                                "brand_name": clean_brand(product.get("brandName", "")),
                                "categories": categories_text,
                                "shop_name": product.get("sellerName", shop_key),
                                "seller_id": product.get("sellerId", ""),
                                "shop_key": shop_key,
                                "location": product.get("location", ""),
                                "current_price": price,
                                "original_price": original_price,
                                # "discount_amount": original_price - price if original_price > price else 0,
                                # "cheapest_sku": product.get("cheapest_sku", ""),
                                "sold_count": parse_sold_count(sold_raw),
                                "sold_count_raw": sold_raw,
                                "rating_score": safe_float(product.get("ratingScore")),
                                "review_count": safe_int(product.get("review")),
                                "in_stock": product.get("inStock", True),
                                "is_sponsored": product.get("isSponsored", False),
                                "product_url": clean_url(product.get("itemUrl", "")),
                                "image_url": clean_url(product.get("image", "")),
                                # "crawl_page": page_no,
                                "source_platform": "Lazada",
                                "collected_at": make_collected_at()
                            })

                    all_products.extend(shop_products)

                    print(f"   >>> Successfully collected {len(shop_products):,} products from '{shop_key}'")

                except Exception as e:
                    print(f"❌ [SKIP] Error in shop '{shop_key}': {e}")
                    failed_shops.append({
                        "shop_key": shop_key,
                        "error": str(e)
                    })
                    continue

        finally:
            await context.close()

    if all_products:
        df_products = pd.DataFrame(all_products)

        dedupe_cols = [
            col for col in ["product_id", "sku_id", "product_url"]
            if col in df_products.columns
        ]

        if dedupe_cols:
            df_products = df_products.drop_duplicates(subset=dedupe_cols)

        filename = Path(OUTPUT_DIR) / f"lazada_products_{make_timestamp()}.xlsx"
        df_products.to_excel(filename, index=False)

        print("\n" + "=" * 60)
        print(f"✅ PRODUCT BY SHOP COMPLETED! Total Products: {len(df_products):,}")
        print(f"💾 File: {filename}")
        print("=" * 60)

    else:
        df_products = pd.DataFrame()
        print("⚠️ No products collected from shops.")

    save_failed_log(failed_shops, "failed_shops")

    return df_products

df_products = await run_product_by_shop(SHOP_URL_KEYS)

📂 [SESSION] Loading Persistent Context: './my_session'

------------------------------------------------------------
[1/2] Processing Shop: mizumi-bomi | URL: https://www.lazada.co.th/mizumi-bomi/?q=All-Products&from=wangpu&langFlag=th&pageTypeId=2
   > Total Products Found: 173 | Total Pages: 5
   > Page 1/5: Found 40 products
   > Page 2/5: Found 40 products
   > Page 3/5: Found 40 products
   > Page 4/5: Found 40 products
   > Page 5/5: Found 13 products
   >>> Successfully collected 173 products from 'mizumi-bomi'

------------------------------------------------------------
[2/2] Processing Shop: ing-on-official | URL: https://www.lazada.co.th/ing-on-official/?q=All-Products&from=wangpu&langFlag=th&pageTypeId=2
   > Total Products Found: 56 | Total Pages: 2
   > Page 1/2: Found 40 products
   > Page 2/2: Found 16 products
   >>> Successfully collected 56 products from 'ing-on-official'

✅ PRODUCT BY SHOP COMPLETED! Total Products: 229
💾 File: collected_data\lazada_products_2026043

In [13]:
pp.pprint(df_products.to_dict(orient="records"))

[ { 'product_id': '3952069815',
    'sku_id': '23751953769',
    'product_name': '[เซ็ต 4 กระปุก] Bomi X Series โบมิ เอ็กซ์ ซีรีส์ '
                    'วิตามินผิวสูตรเข้มข้น อัพผิวสวยแบบเห็นผล',
    'brand_name': 'Bomi',
    'categories': '[10100869, 4087, 8971, 4577]',
    'shop_name': 'MizuMi & Bomi',
    'seller_id': '100152243',
    'shop_key': 'mizumi-bomi',
    'location': 'Samut Prakan',
    'current_price': 1499.0,
    'original_price': 3160.0,
    'sold_count': 3000,
    'sold_count_raw': '3.0K sold',
    'rating_score': 4.969957081545064,
    'review_count': 932,
    'in_stock': True,
    'is_sponsored': False,
    'product_url': 'https://www.lazada.co.th/products/pdp-i3952069815.html',
    'image_url': 'https://th-live-01.slatic.net/p/996b27ae16e1d12c2cf7af3d956f1f5c.jpg',
    'source_platform': 'Lazada',
    'collected_at': '2026-04-30 14:12:01'},
  { 'product_id': '5010918982',
    'sku_id': '21176047921',
    'product_name': '[แพ็ค 3] Bomi Bio S Series เซตทดลอง ดูแลน้ำห

## Product Description

In [ ]:
DETAIL_PRODUCT_URLS = [
    "https://www.lazada.co.th/products/pdp-i5010918982-s21176047921.html",
    "https://www.lazada.co.th/products/pdp-i5430857467-s23060039894.html",
]


async def run_product_detail(product_urls):
    """
    ดึงรายละเอียดสินค้า Lazada

    Output:
    - specs
    - qualification
    - description
    - description images
    """

    all_details = []
    failed_details = []

    async with async_playwright() as p:
        print(f"📂 [SESSION] Loading Persistent Context: '{USER_DATA_DIR}'")

        context, page = await create_browser_context(p, block_images=False)

        try:
            for idx, url in enumerate(product_urls, start=1):
                print("\n" + "-" * 60)
                print(f"[{idx}/{len(product_urls)}] Product Detail URL: {url}")

                try:
                    await page.goto(url, wait_until="domcontentloaded", timeout=90000)
                    await handle_page_check(page, f"Product Detail: {idx}")
                    await page.wait_for_load_state("networkidle", timeout=90000)

                    product_id, product_name, shop_name = await get_product_info(page, url)
                    
                    load_result = await wait_and_scroll_until_detail_loaded(page)
                    if not load_result["loaded"]:
                        print("   ⚠️ Detail content may not be fully loaded.")

                    specs_data = await extract_product_specs(page)
                    qual_data = await extract_product_qualification(page)
                    description_text = await extract_product_description(page)
                    desc_images = await extract_product_description_images(page)

                    print(f"   > Product ID: {product_id}")
                    print(f"   > Product Name: {product_name}")
                    print(f"   > Specs: {len(specs_data)} keys | {', '.join(list(specs_data.keys()))}")
                    print(f"   > Qualifications: {len(qual_data)} keys | {', '.join(list(qual_data.keys()))}")
                    print(f"   > Images: {len(desc_images)}")
                    print(f"   > Description: {description_text[:10]}... ({len(description_text)} chars)")

                    if (len(specs_data) == 0 and len(qual_data) == 0 and len(description_text) == 0 and len(desc_images) == 0):
                        html = await page.content()
                        save_debug_html(html, "debug_empty_detail", product_id)

                    all_details.append({
                        "product_id": product_id,
                        "product_name": product_name,
                        "shop_name": shop_name,
                        "product_url": url,
                        "qualification_info": json.dumps(qual_data, ensure_ascii=False),
                        "all_specs": json.dumps(specs_data, ensure_ascii=False),
                        "description": description_text,
                        "description_images": json.dumps(desc_images, ensure_ascii=False),
                        "source_platform": "Lazada",
                        "collected_at": make_collected_at()
                    })

                    await human_delay()

                except Exception as e:
                    print(f"   ❌ [ERROR] Product detail failed: {e}")
                    failed_details.append({
                        "product_url": url,
                        "error": str(e)
                    })
                    continue

        finally:
            await context.close()

    if all_details:
        df_details = pd.DataFrame(all_details)

        dedupe_cols = [
            col for col in ["product_id", "product_url"]
            if col in df_details.columns
        ]

        if dedupe_cols:
            df_details = df_details.drop_duplicates(subset=dedupe_cols)

        filename = Path(OUTPUT_DIR) / f"lazada_details_{make_timestamp()}.xlsx"
        df_details.to_excel(filename, index=False)

        print("\n" + "=" * 60)
        print(f"✅ PRODUCT DETAIL COMPLETED! Total Products: {len(df_details):,}")
        print(f"💾 File: {filename}")
        print("=" * 60)

    else:
        df_details = pd.DataFrame()
        print("⚠️ No product details collected.")

    save_failed_log(failed_details, "failed_details")

    return df_details

df_details = await run_product_detail(DETAIL_PRODUCT_URLS)

📂 [SESSION] Loading Persistent Context: './my_session'

------------------------------------------------------------
[1/2] Product Detail URL: https://www.lazada.co.th/products/pdp-i5010918982-s21176047921.html
   > Product ID: 5010918982
   > Product Name: [แพ็ค 3] Bomi Bio S Series เซตทดลอง ดูแลน้ำหนัก สุขภาพดี พร้อมเพิ่มกากใย ลำไส้สมดุล
   > Specs: 7 keys | Brand, SKU, Product_License, Product Form, Ingredients, Pack Type, Recommended User
   > Qualifications: 2 keys | License Type, License Code
   > Images: 5
   > Description: Bomi Coffe... (2379 chars)

------------------------------------------------------------
[2/2] Product Detail URL: https://www.lazada.co.th/products/pdp-i5430857467-s23060039894.html
   > Product ID: 5430857467
   > Product Name: [2 Pieces] Lactacid All Day Care 250 Ml. [Twin Pack] Lactacyd All-Day Care 250ml
   > Specs: 8 keys | Brand, SKU, Body Care Benefits, Product Form, Special Claims, Ingredient Preference, Skin Type, Recommended User
   > Qualification

In [41]:
pp.pprint(df_details.to_dict(orient="records"))

[ { 'product_id': '5010918982',
    'product_name': '[แพ็ค 3] Bomi Bio S Series เซตทดลอง ดูแลน้ำหนัก สุขภาพดี '
                    'พร้อมเพิ่มกากใย ลำไส้สมดุล',
    'shop_name': 'MizuMi & Bomi',
    'product_url': 'https://www.lazada.co.th/products/pdp-i5010918982-s21176047921.html',
    'qualification_info': '{"License Type": "TH_FDA_Advertising", "License '
                          'Code": ""}',
    'all_specs': '{"Brand": "Bomi", "SKU": "5010918982_TH-21176047921", '
                 '"Product_License": "20-1-13451-6-0001", "Product Form": '
                 '"Liquid/Powder", "Ingredients": "Fiber", "Pack Type": '
                 '"Multi-pack", "Recommended User": "Adults"}',
    'description': 'Bomi Coffee Bio S\n'
                   'โบมิ คอฟฟี่ ไบโอ เอส\n'
                   'กาแฟคุมน้ำหนัก หอมอร่อย มีพรีไบโอติกส์ไฟเบอร์สูง\n'
                   '✔️ High Prebiotic Fiber '
                   'ไฟเบอร์พรีไบโอติกส์ชั้นดีในปริมาณสูง '
                   'ช่วยเพิ่มปริมาณโพรไบโอติกส์

## Reviews

In [ ]:
REVIEW_PRODUCT_URLS = [
    "https://www.lazada.co.th/products/pdp-i5592245338.html?spm=a2o4m.store_keyword.list.1.4bfd58a6YLqDCI",
    "https://www.lazada.co.th/products/pdp-i5592124928.html?spm=a2o4m.store_keyword.list.3.4bfd58a6YLqDCI",
]


async def run_reviews(product_urls):
    """
    ดึงข้อมูลรีวิวสินค้าจาก Lazada Review API

    Output:
    - return DataFrame
    - export Excel
    """

    all_reviews = []
    failed_products = []

    async with async_playwright() as p:
        print(f"📂 [SESSION] Loading Persistent Context: '{USER_DATA_DIR}'")

        context, page = await create_browser_context(p, block_images=True)

        try:
            for idx, url in enumerate(product_urls, start=1):
                print("\n" + "-" * 60)
                print(f"[{idx}/{len(product_urls)}] Processing Review URL: {url}")

                product_reviews = []

                try:
                    await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                    await handle_page_check(page, f"Product Page: {idx}")

                    product_id, product_name, shop_name = await get_product_info(page, url)
                    print(f"   > Product ID: {product_id}")
                    print(f"   > Product Name: {product_name}")
                    print(f"   > Shop Name: {shop_name}")

                    if product_id == "unknown":
                        raise ValueError("Cannot extract product_id from URL")

                    for page_no in range(1, 501):
                        review_api_url = (
                            "https://my.lazada.co.th/pdp/review/getReviewList"
                            f"?itemId={product_id}"
                            f"&pageSize=50"
                            f"&filter=0"            # 0: All, 1: With Content, 2: With Media
                            f"&sort=0"              # 0: Recent, 1: Rating: High to Low, 2: Rating: Low to High
                            f"&pageNo={page_no}"
                        )

                        await page.goto(review_api_url, wait_until="domcontentloaded", timeout=60000)
                        await human_delay()

                        raw_text = await handle_page_check(page, f"Review API | Product {idx} | Page {page_no}")
                        data = safe_json_loads(raw_text)

                        if not data:
                            print(f"   ! Page {page_no}: JSON Decode Error")
                            break

                        model = data.get("model", {}) or {}
                        items = model.get("items", []) or []

                        if not items:
                            print(f"   > Page {page_no}: No more reviews.")
                            break

                        total_count = safe_int(model.get("ratings", {}).get("reviewCount", 0))

                        current_page_reviews = []
                        for r in items:
                            review_time_raw = r.get("reviewTime", "")

                            current_page_reviews.append({
                                "shop_id": r.get("sellerId", ""),
                                "product_id": product_id,
                                "shop_name": shop_name,
                                "product_name": product_name,
                                "user_name": r.get("buyerName", ""),
                                "rating_score": safe_float(r.get("rating", "")),
                                "review_date_raw": review_time_raw,
                                "review_date": convert_time_interval(review_time_raw),
                                "product_option": r.get("skuInfo", ""),
                                "comment_text": r.get("reviewContent", ""),
                                "source_platform": "Lazada",
                                "product_url": url,
                                # "crawl_page": page_no,
                                "collected_at": make_collected_at()
                            })

                        product_reviews.extend(current_page_reviews)

                        print(f"   > Page {page_no}: Found {len(current_page_reviews)} reviews | Product Total: {len(product_reviews)}")

                        if total_count > 0 and len(product_reviews) >= total_count:
                            print("   > Reached total review count. Stop.")
                            break

                    all_reviews.extend(product_reviews)
                    print(f"   >>> Successfully collected {len(product_reviews):,} reviews for this product.")

                except Exception as e:
                    print(f"   ❌ Error processing review URL: {e}")

                    failed_products.append({
                        "product_url": url, 
                        "error": str(e)
                    })
                    continue

        finally:
            await context.close()

    if all_reviews:
        df_reviews = pd.DataFrame(all_reviews)
        parsed_dates = pd.to_datetime(df_reviews["review_date"],errors="coerce")
        df_reviews["review_date"] = (parsed_dates.dt.strftime("%Y-%m-%d").where(parsed_dates.notna(), df_reviews["review_date"]))

        dedupe_cols = [
            col for col in [
                "product_id",
                "user_name",
                "review_time_raw",
                "comment_text"
            ]
            if col in df_reviews.columns
        ]

        if dedupe_cols:
            df_reviews = df_reviews.drop_duplicates(subset=dedupe_cols)

        filename = Path(OUTPUT_DIR) / f"lazada_reviews_{make_timestamp()}.xlsx"
        df_reviews.to_excel(filename, index=False)

        print("\n" + "=" * 60)
        print(f"✅ REVIEWS COMPLETED! Total Reviews: {len(df_reviews):,}")
        print(f"💾 File: {filename}")
        print("=" * 60)

    else:
        df_reviews = pd.DataFrame()
        print("⚠️ No reviews collected.")

    save_failed_log(failed_products, "failed_reviews")

    return df_reviews

df_reviews = await run_reviews(REVIEW_PRODUCT_URLS)

📂 [SESSION] Loading Persistent Context: './my_session'

------------------------------------------------------------
[1/2] Processing Review URL: https://www.lazada.co.th/products/pdp-i5592245338.html?spm=a2o4m.store_keyword.list.1.4bfd58a6YLqDCI
   > Product ID: 5592245338
   > Product Name: MizuMi UV Bright Body Serum Beige (120g) เซรั่มกันแดดโทนอัพ ผิวไบรท์ทันที ปรับผิวกระจ่างใสขึ้น 1 ระดับสำหรับผิวขาวเหลือง-ผิวสองสี
   > Shop Name: MizuMi & Bomi
   > Page 1: Found 50 reviews | Product Total: 50
   > Page 2: Found 14 reviews | Product Total: 64
   > Page 3: No more reviews.
   >>> Successfully collected 64 reviews for this product.

------------------------------------------------------------
[2/2] Processing Review URL: https://www.lazada.co.th/products/pdp-i5592124928.html?spm=a2o4m.store_keyword.list.3.4bfd58a6YLqDCI
   > Product ID: 5592124928
   > Product Name: [แพ็คคู่] MizuMi UV Bright Body Serum Beige (120g) เซรั่มกันแดดโทนอัพ ผิวไบรท์ทันที ปรับผิวกระจ่างใสขึ้น 1 ระดับสำหรับ

In [39]:
pp.pprint(df_reviews.to_dict(orient="records"))

[ { 'shop_id': 100152243,
    'product_id': '5592245338',
    'shop_name': 'MizuMi & Bomi',
    'product_name': 'MizuMi UV Bright Body Serum Beige (120g) '
                    'เซรั่มกันแดดโทนอัพ ผิวไบรท์ทันที ปรับผิวกระจ่างใสขึ้น 1 '
                    'ระดับสำหรับผิวขาวเหลือง-ผิวสองสี',
    'user_name': 'J***i',
    'rating_score': 5.0,
    'review_date_raw': '05 Apr 2025',
    'review_date': '2025-04-05',
    'product_option': 'Variation3:Beige',
    'comment_text': 'ผิวสีเหลืองใช้อันนี้แล้วผิวสว่างขึ้นแบบธรรมชาติ '
                    'ระวังอย่าทาเยอะเกินค่า\n'
                    ' 💧Moisturizing Effect: ไม่หนักผิวเลย เบาสบาย ',
    'source_platform': 'Lazada',
    'product_url': 'https://www.lazada.co.th/products/pdp-i5592245338.html?spm=a2o4m.store_keyword.list.1.4bfd58a6YLqDCI',
    'collected_at': '2026-04-30 14:52:00'},
  { 'shop_id': 100152243,
    'product_id': '5592245338',
    'shop_name': 'MizuMi & Bomi',
    'product_name': 'MizuMi UV Bright Body Serum Beige (120g) '
  

## Search Keyword

In [ ]:
SEARCH_KEYWORDS = [
    "sunscreen",
    "lipstick",
]

SEARCH_OFFICIAL_ONLY = False

async def run_search_keyword( keywords, official_only=SEARCH_OFFICIAL_ONLY):
    """
    ดึงรายการสินค้าจากหน้า search Lazada ผ่าน ajax=true

    Output:
    - return DataFrame
    - export Excel
    """

    all_products = []
    failed_keywords = []

    async with async_playwright() as p:
        print(f"📂 [SESSION] Loading Persistent Context: '{USER_DATA_DIR}'")

        context, page = await create_browser_context(p, block_images=True)

        try:
            for idx, keyword in enumerate(keywords, start=1):
                keyword_products = []
                encoded_keyword = quote_plus(keyword)
                official_param = "&service=official" if official_only else ""

                print("\n" + "-" * 60)
                print(f"[{idx}/{len(keywords)}] Search Keyword: {keyword} | LazMall: {official_only}")

                base_url = (
                    "https://www.lazada.co.th/catalog/"
                    f"?ajax=true"
                    f"&q={encoded_keyword}"
                    f"&page=1"
                    f"&sort=sold"
                    f"{official_param}"
                )

                try:
                    await page.goto("https://www.lazada.co.th/", wait_until="domcontentloaded", timeout=60000)
                    await asyncio.sleep(1.5)
                    await page.goto(base_url, wait_until="domcontentloaded", timeout=60000)

                    raw_text = await handle_page_check(page, f"Search Meta: {keyword}")
                    data = safe_json_loads(raw_text)

                    if not data:
                        failed_keywords.append({
                            "keyword": keyword,
                            "error": "Empty JSON or JSON Decode Error at search meta"
                        })
                        continue

                    main_info = data.get("mainInfo", {}) or {}

                    total_results = safe_int(main_info.get("totalResults", 0))
                    page_size = safe_int(main_info.get("pageSize", 40), default=40)
                    total_pages = math.ceil(total_results / page_size) if total_results > 0 else 1
                    total_pages = 5

                    print(f"   > Total Items Found: {total_results:,} | Total Pages: {total_pages}")

                    for page_no in range(1, total_pages + 1):
                        current_url = base_url.replace("page=1", f"page={page_no}")

                        await page.goto(current_url,wait_until="domcontentloaded", timeout=60000)
                        await human_delay()

                        page_raw_text = await handle_page_check(page, f"Search: {keyword} | Page {page_no}")
                        page_json = safe_json_loads(page_raw_text)

                        if not page_json:
                            print(f"   ! Page {page_no}: JSON Decode Error")
                            break

                        items = (page_json.get("mods", {}).get("listItems", [])) or []
                        print(f"   > Page {page_no}/{total_pages}: Found {len(items)} items")

                        if not items:
                            break

                        for product in items:
                            price = safe_float(product.get("price"))
                            original_price = safe_float(product.get("originalPrice"))
                            sold_raw = product.get("itemSoldCntShow", "0")

                            icons = product.get("icons", [])
                            if not isinstance(icons, list):
                                icons = []

                            # LazMall จะมี icon ที่มี bizType = lazMall
                            is_mall = any(isinstance(icon, dict) and icon.get("bizType") == "lazMall" for icon in icons)

                            keyword_products.append({
                                "product_id": product.get("itemId", ""),
                                "sku_id": product.get("skuId", ""),
                                "product_name": product.get("name", ""),
                                "brand_name": clean_brand(product.get("brandName", "")),
                                "discount_price": price,
                                "original_price": original_price,
                                "sold_count": parse_sold_count(sold_raw),
                                "sold_count_raw": sold_raw,
                                "rating_score": safe_float(product.get("ratingScore")),
                                "review_count": safe_int(product.get("review")),
                                "shop_name": product.get("sellerName", ""),
                                "seller_id": product.get("sellerId", ""),
                                "is_mall": is_mall,
                                "location": product.get("location", ""),
                                "is_sponsored": product.get("isSponsored", False),
                                "product_url": clean_url(product.get("itemUrl", "")),
                                "image_url": clean_url(product.get("image", "")),
                                "search_keyword": keyword,
                                "source_platform": "Lazada",
                                # "crawl_page": page_no,
                                "collected_at": make_collected_at()
                            })

                    all_products.extend(keyword_products)

                    print(f"   >>> Successfully collected {len(keyword_products):,} items for keyword '{keyword}'.")

                except Exception as e:
                    print(f"❌ [SKIP] Error in keyword '{keyword}': {e}")

                    failed_keywords.append({
                        "keyword": keyword,
                        "error": str(e)
                    })

                    continue

        finally:
            await context.close()

    if all_products:
        df_search = pd.DataFrame(all_products)

        dedupe_cols = [
            col for col in [
                "search_keyword",
                "product_id",
                "sku_id",
                "product_url"
            ]
            if col in df_search.columns
        ]

        if dedupe_cols:
            df_search = df_search.drop_duplicates(subset=dedupe_cols)

        filename = Path(OUTPUT_DIR) / f"lazada_search_{make_timestamp()}.xlsx"
        df_search.to_excel(filename, index=False)

        print("\n" + "=" * 60)
        print(f"✅ SEARCH KEYWORD COMPLETED! Total Products: {len(df_search):,}")
        print(f"💾 File: {filename}")
        print("=" * 60)

    else:
        df_search = pd.DataFrame()
        print("⚠️ No search products collected.")

    save_failed_log(failed_keywords, "failed_keywords")

    return df_search

df_search = await run_search_keyword(SEARCH_KEYWORDS, official_only=False)

📂 [SESSION] Loading Persistent Context: './my_session'

------------------------------------------------------------
[1/2] Search Keyword: sunscreen | LazMall: False
   > Total Items Found: 3,734 | Total Pages: 5
   > Page 1/5: Found 40 items
   > Page 2/5: Found 40 items
   > Page 3/5: Found 40 items
   > Page 4/5: Found 40 items
   > Page 5/5: Found 40 items
   >>> Successfully collected 200 items for keyword 'sunscreen'.

------------------------------------------------------------
[2/2] Search Keyword: lipstick | LazMall: False
   > Total Items Found: 3,770 | Total Pages: 5
   > Page 1/5: Found 40 items
   > Page 2/5: Found 40 items
   > Page 3/5: Found 40 items
   > Page 4/5: Found 40 items
   > Page 5/5: Found 40 items
   >>> Successfully collected 200 items for keyword 'lipstick'.

✅ SEARCH KEYWORD COMPLETED! Total Products: 400
💾 File: collected_data\lazada_search_20260430_145102.xlsx


In [35]:
pp.pprint(df_search.to_dict(orient="records"))

[ { 'product_id': '3827008221',
    'sku_id': '14596833883',
    'product_name': '[Available in Packs of 2 and 4] Mizumi Uv Bright Body '
                    'Serum (180 ml) Sunscreen Serum for the Body, Light and '
                    'Comfortable on the Skin, Gentle Fragrance, Protects the '
                    'Skin from the Sun and Pollution.',
    'brand_name': 'MizuMi',
    'discount_price': 359.0,
    'original_price': 780.0,
    'sold_count': 39400,
    'sold_count_raw': '39.4K sold',
    'rating_score': 4.989144736842105,
    'review_count': 12160,
    'shop_name': 'MizuMi & Bomi',
    'seller_id': '100152243',
    'is_mall': True,
    'location': 'Samut Prakan',
    'is_sponsored': False,
    'product_url': 'https://www.lazada.co.th/products/pdp-i3827008221.html',
    'image_url': 'https://sg-test-11.slatic.net/p/d163cc1a51bc567c24a583e27bff4647.jpg',
    'search_keyword': 'sunscreen',
    'source_platform': 'Lazada',
    'collected_at': '2026-04-30 14:50:35'},
  { 'product_i